# 03 — Analyst decrypts only released aggregate results

## Goal

Try the in-memory CKKS PRE boundary added in SDK `0.4.1.dev0`. The analyst receives a different secret key and can decrypt released `sum`, `mean`, and `variance`, but not the owner input ciphertext.

## Setup

Run this on the Linux notebook image with the OpenFHE extra installed. This first trial intentionally does not persist analyst keys or PRE re-keys.

In [ ]:
from he_sdk import HESession, ResultReleaseError

VALUES = [10.0, 20.0, 30.0]

## Steps

Create an owner ciphertext and a distinct analyst key pair in the same CKKS context. Then compute and release only the approved aggregates.

In [ ]:
owner = HESession.create(backend="openfhe")
encrypted_input = owner.encrypt(VALUES)
analyst = owner.create_result_recipient()

encrypted_results = {
    "sum": owner.sum(encrypted_input),
    "mean": owner.mean(encrypted_input),
    "variance": owner.variance(encrypted_input),
}
released_results = {
    operation: owner.release_result(result, to=analyst)
    for operation, result in encrypted_results.items()
}

## Checks

The first check must report that the input is blocked. The analyst then sees only three scalar aggregates.

In [ ]:
try:
    analyst.decrypt(encrypted_input)
except ResultReleaseError:
    print("PASS: owner input is not analyst-decryptable")
else:
    raise AssertionError("analyst unexpectedly decrypted owner input")

observed = {
    operation: analyst.decrypt(result)
    for operation, result in released_results.items()
}
print(observed)
assert abs(observed["sum"] - 60.0) < 0.1
assert abs(observed["mean"] - 20.0) < 0.1
assert abs(observed["variance"] - (200.0 / 3.0)) < 0.1
owner.close()

## Next steps

Keep notebook 02 compute-only. Move `release_result()` into an isolated service before persisting analyst keys or released ciphertexts. Do not put a PRE re-key in the shared workspace or PostgreSQL.